# DeepTruth — Full Retrain V4 (Anti-Overfit)
Fixes from V3: JPEG compression augmentation, dropout=0.4, phase 3 reduced to 3 epochs.
- Phase 1 (3 epochs): head only
- Phase 2 (4 epochs): unfreeze freq + srm + gram
- Phase 3 (3 epochs): unfreeze all streams with tiny LR
- JPEG compression augmentation simulates phone/WhatsApp photos during training

In [ ]:
# ── 1. Mount Drive & setup ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, random, io
DRIVE_DIR = '/content/drive/MyDrive/deeptruth'

shutil.copy(f'{DRIVE_DIR}/model_arch.py', '/content/model_arch.py')
print('model_arch.py copied')

import IPython
IPython.display.display(IPython.display.Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
'''))

In [ ]:
# ── 2. Install packages ────────────────────────────────────────────────
!pip install -q timm transformers kaggle

In [ ]:
# ── 3. Kaggle credentials ──────────────────────────────────────────────
import os, json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

kaggle_key = "PASTE_YOUR_KEY_HERE"
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({"username": "tameemelfakharany", "key": kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle ready')

In [ ]:
# ── 4. Download & extract datasets ────────────────────────────────────
import zipfile
from pathlib import Path

print('Extracting processed_faces.zip...')
with zipfile.ZipFile(f'{DRIVE_DIR}/processed_faces.zip', 'r') as z:
    z.extractall('/content/processed_faces')
print('processed_faces extracted')

print('Downloading VGGFace2...')
!kaggle datasets download -d zenbot99/vggface2-hq-cropped -p /content/vggface2 --unzip -q
print('VGGFace2 done')

In [ ]:
# ── 5. Collect datasets ────────────────────────────────────────────────
EXTS = {'.jpg', '.jpeg', '.png'}
random.seed(42)

base = '/content/processed_faces/processed_faces'

ffhq_imgs      = [str(p) for p in Path(f'{base}/real').rglob('*') if p.suffix.lower() in EXTS]
extra_real     = [str(p) for p in Path(f'{DRIVE_DIR}/extra_fakes/Real').rglob('*') if p.suffix.lower() in EXTS]
vgg_imgs       = [str(p) for p in Path('/content/vggface2').rglob('*.jpg')]
processed_fake = [str(p) for p in Path(f'{base}/fake').rglob('*') if p.suffix.lower() in EXTS]
extra_fake     = [str(p) for p in Path(f'{DRIVE_DIR}/extra_fakes').rglob('*')
                  if p.suffix.lower() in EXTS and 'Real' not in str(p)]

all_real = ffhq_imgs + extra_real + vgg_imgs
all_fake = processed_fake + extra_fake
random.shuffle(all_real)
random.shuffle(all_fake)

print(f'Real — FFHQ: {len(ffhq_imgs)}, extra_real: {len(extra_real)}, VGGFace2: {len(vgg_imgs)}')
print(f'Total real: {len(all_real)}')
print(f'Total fake: {len(all_fake)}')

In [ ]:
# ── 6. Dataset & DataLoader ────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Simulates phone/WhatsApp compression on real images during training
class RandomJPEGCompression:
    def __call__(self, img):
        quality = random.randint(20, 85)
        buffer  = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    RandomJPEGCompression(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.05),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FaceDataset(Dataset):
    def __init__(self, real_paths, fake_paths, transform=None):
        self.items = [(p, 0) for p in real_paths] + [(p, 1) for p in fake_paths]
        random.shuffle(self.items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            return torch.zeros(3, 224, 224), label

def split(lst, ratio=0.9):
    n = int(len(lst) * ratio)
    return lst[:n], lst[n:]

real_train, real_val = split(all_real)
fake_train, fake_val = split(all_fake)

train_ds = FaceDataset(real_train, fake_train, transform=train_tf)
val_ds   = FaceDataset(real_val,   fake_val,   transform=val_tf)

labels  = [item[1] for item in train_ds.items]
n_real  = labels.count(0)
n_fake  = labels.count(1)
weights = [1.0/n_real if l == 0 else 1.0/n_fake for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=100000, replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} ({n_real} real, {n_fake} fake)')
print(f'Val:   {len(val_ds)}')
print(f'Batches per epoch: {len(train_loader)}')

In [ ]:
# ── 7. Load stage1_warmup_best checkpoint ─────────────────────────────
import sys, time
sys.path.insert(0, '/content')
from model_arch import DeepTruthHybridV2, DeepTruthLoss

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_PATH = f'{DRIVE_DIR}/checkpoints/deeptruth_v4_best.pth'
CKPT_PATH = f'{DRIVE_DIR}/checkpoints/stage1_warmup_best.pth'
print(f'Device: {DEVICE}')

ckpt = torch.load(CKPT_PATH, map_location='cpu')
num_fake_types = ckpt.get('num_fake_types', 10)

torch.manual_seed(42)
# dropout=0.4 (increased from 0.3) to reduce overfitting
model = DeepTruthHybridV2(num_fake_types=num_fake_types, dropout=0.4)

remap = {
    'stream1_clip': 'clip_stream', 'stream2_effnet': 'effnet_stream',
    'stream3_freq': 'freq_stream', 'stream4_srm':  'srm_stream',
    'stream5_gram': 'gram_stream', 'fusion':        'image_fusion',
    'head':         'image_head',  'binary_out':    'image_binary_out',
    'type_out':     'image_type_out',
}
state = {}
for k, v in ckpt['model_state'].items():
    new_k = k
    for old, new in remap.items():
        if k.startswith(old + '.'):
            new_k = new + k[len(old):]
            break
    state[new_k] = v

missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded: {len(state) - len(unexpected)}/{len(state)} keys')
print(f'Starting metrics: {ckpt.get("metrics", {})}')
model = model.to(DEVICE)

best_avg = 0.0
print(f'best_avg starts at {best_avg:.4f}')

In [ ]:
# ── 8. Shared evaluate + save functions ────────────────────────────────

def evaluate():
    model.eval()
    correct_real = total_real = correct_fake = total_fake = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            preds = out['fake_logit'].argmax(dim=1)
            real_mask = labels == 0
            fake_mask = labels == 1
            correct_real += (preds[real_mask] == 0).sum().item()
            total_real   += real_mask.sum().item()
            correct_fake += (preds[fake_mask] == 1).sum().item()
            total_fake   += fake_mask.sum().item()
    real_acc = correct_real / max(total_real, 1)
    fake_acc = correct_fake / max(total_fake, 1)
    return real_acc, fake_acc, (real_acc + fake_acc) / 2


def save_if_best(real_acc, fake_acc, avg_acc, phase):
    global best_avg
    if avg_acc > best_avg:
        best_avg = avg_acc
        torch.save({
            'model_state':    model.state_dict(),
            'num_fake_types': num_fake_types,
            'fake_type_map':  ckpt.get('fake_type_map', {}),
            'phase':          phase,
            'metrics':        {'real_acc': real_acc, 'fake_acc': fake_acc, 'avg_acc': avg_acc},
        }, SAVE_PATH)
        print(f'  ✓ Saved best checkpoint (avg={avg_acc*100:.1f}%)')
        return True
    return False


def train_phase(phase_name, phase_num, epochs, optimizer, scheduler):
    criterion = DeepTruthLoss(type_weight=0.0, gamma=2.0, label_smoothing=0.1)
    print(f'\n═══ {phase_name} ═══')
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        t0 = time.time()

        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out['binary_logit'], labels)

            if torch.isnan(loss):
                print(f'  NaN at batch {batch_idx}, skipping')
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

            if (batch_idx + 1) % 100 == 0:
                print(f'  [{batch_idx+1}/{len(train_loader)}] loss={loss.item():.4f}')

        scheduler.step()
        real_acc, fake_acc, avg_acc = evaluate()
        elapsed = time.time() - t0
        print(f'Epoch {epoch}/{epochs} | loss={total_loss/len(train_loader):.4f} | '
              f'real={real_acc*100:.1f}% fake={fake_acc*100:.1f}% avg={avg_acc*100:.1f}% | {elapsed:.0f}s')

        saved = save_if_best(real_acc, fake_acc, avg_acc, phase=phase_num)
        if not saved:
            print(f'  No improvement (global best={best_avg*100:.1f}%)')

    print(f'{phase_name} done. Global best so far: {best_avg*100:.1f}%')

In [ ]:
# ── 9. PHASE 1: Head only (3 epochs) ──────────────────────────────────
for p in model.parameters():
    p.requires_grad = False
for m in [model.image_head, model.image_binary_out, model.image_fusion]:
    for p in m.parameters():
        p.requires_grad = True

optimizer1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-4, weight_decay=1e-4
)
scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=3, eta_min=1e-6)

train_phase('Phase 1 — Head Only', phase_num=1, epochs=3,
            optimizer=optimizer1, scheduler=scheduler1)

In [ ]:
# ── 10. PHASE 2: Lightweight streams (4 epochs) ────────────────────────
for m in [model.freq_stream, model.srm_stream, model.gram_stream]:
    for p in m.parameters():
        p.requires_grad = True
for m in [model.clip_stream, model.effnet_stream]:
    for p in m.parameters():
        p.requires_grad = False

optimizer2 = torch.optim.AdamW([
    {'params': model.image_head.parameters(),       'lr': 2e-4},
    {'params': model.image_binary_out.parameters(), 'lr': 2e-4},
    {'params': model.image_fusion.parameters(),     'lr': 2e-4},
    {'params': model.freq_stream.parameters(),      'lr': 5e-5},
    {'params': model.srm_stream.parameters(),       'lr': 5e-5},
    {'params': model.gram_stream.parameters(),      'lr': 5e-5},
], weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=4, eta_min=1e-7)

train_phase('Phase 2 — Lightweight Streams', phase_num=2, epochs=4,
            optimizer=optimizer2, scheduler=scheduler2)

In [ ]:
# ── 11. PHASE 3: All streams (3 epochs — reduced to prevent overfit) ───
for p in model.parameters():
    p.requires_grad = True

optimizer3 = torch.optim.AdamW([
    {'params': model.clip_stream.parameters(),      'lr': 5e-6},
    {'params': model.effnet_stream.parameters(),    'lr': 1e-5},
    {'params': model.freq_stream.parameters(),      'lr': 2e-5},
    {'params': model.srm_stream.parameters(),       'lr': 2e-5},
    {'params': model.gram_stream.parameters(),      'lr': 2e-5},
    {'params': model.image_fusion.parameters(),     'lr': 5e-5},
    {'params': model.image_head.parameters(),       'lr': 5e-5},
    {'params': model.image_binary_out.parameters(), 'lr': 5e-5},
    {'params': model.image_type_out.parameters(),   'lr': 5e-5},
], weight_decay=2e-4)
scheduler3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=3, eta_min=1e-7)

train_phase('Phase 3 — All Streams Unfrozen', phase_num=3, epochs=3,
            optimizer=optimizer3, scheduler=scheduler3)

print(f'\nAll phases done. Best avg: {best_avg*100:.1f}%')
print(f'Best model saved to: {SAVE_PATH}')

In [ ]:
# ── 12. Copy to Drive root ─────────────────────────────────────────────
import shutil

FINAL_PATH = f'{DRIVE_DIR}/deeptruth_v4_final.pth'
shutil.copy(SAVE_PATH, FINAL_PATH)
print(f'Copied to: {FINAL_PATH}')

saved = torch.load(SAVE_PATH, map_location='cpu')
print('Best metrics:', saved['metrics'])
print('Best phase:', saved['phase'])
size_mb = os.path.getsize(SAVE_PATH) / 1024 / 1024
print(f'Size: {size_mb:.0f} MB')

In [ ]:
# ── 13. Test on images — loads BEST checkpoint before testing ──────────
import cv2
import numpy as np
from google.colab import files
from torchvision import transforms as T
from PIL import Image as PILImage

best_ckpt = torch.load(SAVE_PATH, map_location='cpu')
model.load_state_dict(best_ckpt['model_state'])
model.eval()
print(f'Loaded best model — phase {best_ckpt["phase"]}, metrics: {best_ckpt["metrics"]}')

test_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def preprocess_face(img_rgb):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces   = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40))
    h, w    = img_rgb.shape[:2]
    if len(faces) == 0:
        side = min(h, w)
        crop = img_rgb[(h-side)//2:(h-side)//2+side, (w-side)//2:(w-side)//2+side]
    else:
        fx, fy, fw, fh = max(faces, key=lambda f: f[2]*f[3])
        pad_x, pad_y   = int(fw*0.3), int(fh*0.3)
        crop = img_rgb[max(0,fy-pad_y):min(h,fy+fh+pad_y), max(0,fx-pad_x):min(w,fx+fw+pad_x)]
    return cv2.resize(crop, (224, 224))

def test_image(path):
    img    = np.array(PILImage.open(path).convert('RGB'))
    crop   = preprocess_face(img)
    tensor = test_tf(PILImage.fromarray(crop)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out   = model(tensor)
        probs = torch.softmax(out['fake_logit'], dim=1)[0]
        real_prob = float(probs[0]) * 100
        fake_prob = float(probs[1]) * 100
    verdict = 'FAKE' if fake_prob > 50 else 'REAL'
    print(f'{path}: Real={real_prob:.1f}%  Fake={fake_prob:.1f}%  → {verdict}')

print('Upload test images (mix of real phone photos AND fake/AI images):')
uploaded = files.upload()
for filename in uploaded:
    test_image(filename)

In [ ]:
# ── 14. Download best model ────────────────────────────────────────────
from google.colab import files
files.download(SAVE_PATH)
print('Download started.')